# 06 · Loss Functions

**Learning objectives**
1. Compute MSE, binary cross-entropy (BCE) and categorical cross-entropy (CCE).
2. Verify numerically that the gradient of BCE with a sigmoid output is $p-y$.
3. Compute cross-entropy from logits in a numerically stable way.

Modules: `src/losses.py`, `tests/test_activations_losses.py`.

## 1. Theory

A loss turns the gap between prediction and target into one number. Training minimizes its average.

- **MSE** (regression): $\ell = (\hat y - y)^2$. Gradient $2(\hat y - y)$.
- **BCE** (binary, $p=\1(z)$): $\ell = -[y\log p + (1-y)\log(1-p)]$. Gradient with respect to $z$: $p - y$.
- **CCE** (multi-class, one-hot $y$): $\ell = -\log p_t$. Gradient with respect to logits: $\mathbf{p} - \mathbf{y}$.

The cancellation in BCE and CCE is why sigmoid and softmax pair with these losses: the output derivative does not shrink in the wrong direction.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

from losses import mse, bce, cce_from_logits, log_softmax
from activations import sigmoid, softmax

print("MSE([1,0],[0.8,0.3]) =", round(mse(np.array([1.0, 0.0]), np.array([0.8, 0.3])), 4))
print("BCE(y=1, p=0.8) =", round(float(bce(np.array(1.0), np.array(0.8))), 4))
print("BCE(y=1, p=0.1) =", round(float(bce(np.array(1.0), np.array(0.1))), 4))
print("BCE(y=0, p=0.1) =", round(float(bce(np.array(0.0), np.array(0.1))), 4))

MSE([1,0],[0.8,0.3]) = 0.065
BCE(y=1, p=0.8) = 0.2231
BCE(y=1, p=0.1) = 2.3026
BCE(y=0, p=0.1) = 0.1054


## 2. Gradient check: BCE with sigmoid

Compare the analytic gradient $p-y$ with a central finite difference at several logits.

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

eps = 1e-6
rows = []
for z0 in [-2.0, 0.3, 1.5]:
    for y in [0.0, 1.0]:
        f = lambda zz: float(bce(np.array(y), sigmoid(np.array(zz))))
        numeric = (f(z0 + eps) - f(z0 - eps)) / (2 * eps)
        analytic = float(sigmoid(np.array(z0))) - y
        rows.append((z0, y, numeric, analytic, abs(numeric - analytic)))
print(f"{'z':>6} {'y':>4} {'numeric':>12} {'p - y':>12} {'|diff|':>10}")
for r in rows:
    print(f"{r[0]:6.2f} {r[1]:4.0f} {r[2]:12.8f} {r[3]:12.8f} {r[4]:10.2e}")

     z    y      numeric        p - y     |diff|
 -2.00    0   0.11920292   0.11920292   3.06e-11
 -2.00    1  -0.88079708  -0.88079708   3.78e-11
  0.30    0   0.57444252   0.57444252   2.03e-11
  0.30    1  -0.42555748  -0.42555748   8.50e-12
  1.50    0   0.81757448   0.81757448   2.96e-10
  1.50    1  -0.18242552  -0.18242552   4.99e-11


## 3. Categorical cross-entropy from logits

Logits $\mathbf{z}=(2,1,0.1)$ with true class $t=0$. The gradient with respect to $\mathbf{z}$ is $\mathbf{p}-\mathbf{y}$.

In [3]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

z = np.array([2.0, 1.0, 0.1])
t = 0
print("loss =", round(cce_from_logits(z, t), 4))
p = softmax(z)
print("p =", p.round(4), " gradient p - onehot(t) =", (p - np.eye(3)[t]).round(4))
print("loss with logits shifted by 1000 (same answer):", round(cce_from_logits(z + 1000.0, t), 4))
big = np.array([1002.0, 1001.0, 1000.0])
print("loss with logits 1002/1001/1000 (stable):", round(cce_from_logits(big, 0), 4))

loss = 0.417
p = [0.659  0.2424 0.0986]  gradient p - onehot(t) = [-0.341   0.2424  0.0986]
loss with logits shifted by 1000 (same answer): 0.417
loss with logits 1002/1001/1000 (stable): 0.4076


## 4. Interpretation

- MSE measures squared distance and is sensitive to outliers. It is a poor choice for probabilities from sigmoid or softmax, because the gradient gets multiplied by a small derivative.
- BCE and CCE are the negative log-likelihood of the label under the model. Confident wrong answers cost a lot.
- Compute cross-entropy from logits, not from probabilities, to avoid overflow and $\log 0$.

## 5. Exercises

1. For $y=2$ and $\hat y=0.5$, compute the squared error and its gradient by hand.
2. For logits $(0,0,0)$ and $t=1$, the loss is $\log 3$. Verify it in code and explain why.
3. Explain why sigmoid + MSE learns slowly when the prediction is confidently wrong.